# Step 08 — Tree of Thought

🇬🇧 **English** (this notebook) · 🇩🇪 Deutsch — not yet available.

Instead of committing to one line of reasoning, ask several "experts" to reason in parallel: each writes down one step, shares it with the group, then all move on to the next step together. Any expert whose reasoning turns out to be wrong drops out along the way.

## Learning objective

By the end of this notebook, you will:

- Understand what "tree of thought" prompting means, and how it differs from a single chain of reasoning
- Have run a lightweight, single-prompt simulation of multiple reasoning paths
- Be able to judge whether exploring several angles surfaces something a single reasoning chain misses, or just adds discussion for its own sake

## Prerequisites

- [Step 07 — Chain of Thought](step_07_chain_of_thought.ipynb) completed — this notebook directly compares against it
- The same `.env` setup as the previous steps

## Background

Chain-of-thought (Step 07) commits to a single line of reasoning from start to finish. Tree-of-thought instead explores **multiple** reasoning paths, with the ability to compare them against each other and drop a path partway through if it turns out to be wrong:

> Yao, S., et al. (2023). *Tree of Thoughts: Deliberate Problem Solving with Large Language Models*. NeurIPS 2023. [arXiv:2305.10601](https://arxiv.org/abs/2305.10601)

The original paper implements this as an actual search-and-backtrack procedure across separate model calls. This notebook uses a much lighter version instead: one single prompt that asks the model to simulate several "experts" reasoning step by step in parallel, comparing notes as they go.

## How this works

Run the Setup cell once, then the exercise cell:

1. **Setup** loads your API keys and creates the `llm` object, same as before.
2. **The exercise cell** builds one `user_message` that instructs the model to *role-play* `num_experts` separate experts, each thinking one step at a time, sharing that step with the "group," and dropping out if their own reasoning turns out to be wrong.
3. This is still a single `llm.call(...)` — the "multiple experts" only exist inside the model's one response, simulated through the instruction, not as separate calls.

In [ ]:
import os

# CrewAI's telemetry tries to reach its backend over the network on import;
# on a restricted/firewalled connection this can hang for a long time with no
# error. Disable it before crewai is imported.
os.environ.setdefault('CREWAI_DISABLE_TELEMETRY', 'true')

from dotenv import load_dotenv
from crewai import LLM
from IPython.display import Markdown, display

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

In [ ]:
# ── User message — ask several "experts" to reason step by step in parallel,
# sharing and checking each other's steps, dropping any expert who goes wrong ──
num_experts = 3
question = (
    "What are the main compliance risks the EU AI Act creates for a B2B SaaS "
    "company that uses LLMs in its product, and how should they prioritize "
    "addressing them?"
)

user_message = (
    f"Imagine {num_experts} different experts are answering this question. "
    f"All experts will write down 1 step of their thinking, then share it "
    f"with the group. Then all experts will go on to the next step, etc. "
    f"If any expert realizes they're wrong at any point then they leave. "
    f"The question is '{question}' Make sure to discuss the results."
)

output = llm.call(messages=[{"role": "user", "content": user_message}])
display(Markdown(output))

## Your task

1. Run the cells (Setup first, then the exercise cell).

2. Compare to Step 07 — do the "experts" actually disagree and drop out, or do they converge immediately on the same prioritization? Does exploring multiple angles surface a risk the single chain-of-thought answer in Step 07 missed?

3. Try changing `num_experts` to 2 or 5 and re-run. Does more "experts" produce a meaningfully richer discussion, or just more repetition?

4. Swap `question` for one about your own team's topic.

5. Note what you observed — it's evidence for `EVALUATION.md`'s Section 4.3 (Prompting Techniques: Tree of Thought).

## Shortcomings

This is a simulation, not real parallel search — a single model call role-playing multiple "experts" can still converge on one shared blind spot, since it's the same underlying model reasoning about all of them (task 2 above asks you to check whether that happened). It's also noticeably more expensive in tokens and slower to read than a single chain of thought, for a benefit that isn't guaranteed to show up on every question.

This closes out the prompting-techniques steps. [Step 09](step_09_single_agent.ipynb) moves from prompting alone to CrewAI's `Agent` — reasoning that can also *act*, not just answer.

## Resources for further reading

- Yao, S., et al. (2023). *Tree of Thoughts: Deliberate Problem Solving with Large Language Models*. NeurIPS 2023. [arXiv:2305.10601](https://arxiv.org/abs/2305.10601)
- Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. NeurIPS 2022. [arXiv:2201.11903](https://arxiv.org/abs/2201.11903) — useful to re-read after seeing tree-of-thought, as the direct point of comparison.